In [1]:
import sys
import os
import json
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

import time
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import config
import src.pipeline.rule_generator as rule_gen
import src.pipeline.prolog_generator as prolog_gen_single
import src.pipeline.prolog_composer as prolog_gen_multi

---

## 1. Setup & Configs

In [2]:
RULEBOOK_DIR = Path("../testing/rulebooks")

RULEBOOK_FILES = [
    "checkers.txt",
    "chess.txt",
    "connect_four.txt",
    "nim.txt",
    "reversi.txt",
    "tic_tac_toe.txt"
]

RUNS_PER_GAME = 3

RESULTS_DIR = Path("../testing/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def sanitize_filename(name: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in name)


CONFIGS = [
#   {
#       "name": "qwen3-coder:480b-cloud + single prolog",
#       "BACKEND_PROLOG_GENERATOR": "ollama",
#       "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
#       "PROLOG_USE_MULTISTAGE":    False,
#       "PROLOG_USE_DESIGN_PLAN":   False
#   },
#   {
#       "name": "qwen3-coder:480b-cloud + multi prolog",
#       "BACKEND_PROLOG_GENERATOR": "ollama",
#       "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
#       "PROLOG_USE_MULTISTAGE":    True,
#       "PROLOG_USE_DESIGN_PLAN":   False
#   },
#   {
#       "name": "qwen3-coder:480b-cloud + single prolog + design plan",
#       "BACKEND_PROLOG_GENERATOR": "ollama",
#       "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
#       "PROLOG_USE_MULTISTAGE":    False,
#       "PROLOG_USE_DESIGN_PLAN":   True
#   },
#   {
#       "name": "qwen3-coder:480b-cloud + multi prolog + design plan",
#       "BACKEND_PROLOG_GENERATOR": "ollama",
#       "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
#       "PROLOG_USE_MULTISTAGE":    True,
#       "PROLOG_USE_DESIGN_PLAN":   True
#   },
    {
        "name": "o4-mini + single prolog",
        "BACKEND_PROLOG_GENERATOR": "openai",
        "MODEL_PROLOG_GENERATOR":   "o4-mini",
        "PROLOG_USE_MULTISTAGE":    False,
        "PROLOG_USE_DESIGN_PLAN":   False
    },
    {
        "name": "o4-mini + multi prolog",
        "BACKEND_PROLOG_GENERATOR": "openai",
        "MODEL_PROLOG_GENERATOR":   "o4-mini",
        "PROLOG_USE_MULTISTAGE":    True,
        "PROLOG_USE_DESIGN_PLAN":   False
    },
    {
        "name": "o4-mini + single prolog + design plan",
        "BACKEND_PROLOG_GENERATOR": "openai",
        "MODEL_PROLOG_GENERATOR":   "o4-mini",
        "PROLOG_USE_MULTISTAGE":    False,
        "PROLOG_USE_DESIGN_PLAN":   True
    },
    {
        "name": "o4-mini + multi prolog + design plan",
        "BACKEND_PROLOG_GENERATOR": "openai",
        "MODEL_PROLOG_GENERATOR":   "o4-mini",
        "PROLOG_USE_MULTISTAGE":    True,
        "PROLOG_USE_DESIGN_PLAN":   True
    },
#    {
#        "name": "o3 + single prolog",
#        "BACKEND_PROLOG_GENERATOR": "openai",
#        "MODEL_PROLOG_GENERATOR":   "o3",
#        "PROLOG_USE_MULTISTAGE":    False,
#        "PROLOG_USE_DESIGN_PLAN":   False
#    },
#    {
#        "name": "o3 + multi prolog",
#        "BACKEND_PROLOG_GENERATOR": "openai",
#        "MODEL_PROLOG_GENERATOR":   "o3",
#        "PROLOG_USE_MULTISTAGE":    True,
#        "PROLOG_USE_DESIGN_PLAN":   False
#    },
#    {
#        "name": "o3 + single prolog + design plan",
#        "BACKEND_PROLOG_GENERATOR": "openai",
#        "MODEL_PROLOG_GENERATOR":   "o3",
#        "PROLOG_USE_MULTISTAGE":    False,
#        "PROLOG_USE_DESIGN_PLAN":   True
#    },
#    {
#        "name": "o3 + multi prolog + design plan",
#        "BACKEND_PROLOG_GENERATOR": "openai",
#        "MODEL_PROLOG_GENERATOR":   "o3",
#        "PROLOG_USE_MULTISTAGE":    True,
#        "PROLOG_USE_DESIGN_PLAN":   True
#    },
]


---

## 2. Benchmark runner

In [3]:
def apply_config(cfg : dict):
    for k, v in cfg.items():
        if k != "name" and hasattr(config, k):
            setattr(config, k, v)


def run_single(rulebook: str, cfg_name: str, structured: dict, run_id: int):
    game_name = rulebook.removesuffix(".txt")
    safe_cfg  = sanitize_filename(cfg_name)

    result = {
        "rulebook_file":    rulebook,
        "game_name":        game_name,
        "config":           cfg_name,
        "success":          False,
        "retries":          0,
        "failed_checks":    [],
        "duration":         0.0
    }

    t0 = time.time()
    last_code = [None]

    try:
        prolog_gen = prolog_gen_multi if config.PROLOG_USE_MULTISTAGE else prolog_gen_single
        attempt_count = [0]
        last_errors = [[]]

        if prolog_gen is prolog_gen_single:
            original_validate = prolog_gen_single._validate_prolog

            def tracking_validate(code):
                last_code[0] = code  # ← hinzu
                attempt_count[0] += 1
                valid, errors = original_validate(code)
                last_errors[0] = errors
                return valid, errors

            prolog_gen_single._validate_prolog = tracking_validate

        code, design_plan = prolog_gen.generate_prolog(structured)

        if prolog_gen is prolog_gen_single:
            prolog_gen_single._validate_prolog = original_validate
            result["retries"] = attempt_count[0] - 1
        else:
            result["retries"] = 0

        if code is None:
            checks = last_errors[0] if prolog_gen is prolog_gen_single else []
            result["failed_checks"] = checks
            result["error_stage"]   = "; ".join(checks) if checks else "max retries exceeded (no error detail - multi-stage generation)"
            return result

        result["success"] = True

    except Exception as e:
        result["error_stage"] = f"exception: {type(e).__name__}: {e}"

    finally:
        result["duration"] = round(time.time() - t0, 1)

        # Save prolog (if generated)
        final_code = code if result["success"] else last_code[0]
        if final_code is not None:
            pl_dir = RESULTS_DIR / "prolog" / game_name
            pl_dir.mkdir(parents=True, exist_ok=True)
            (pl_dir / f"{run_id}_{safe_cfg}.pl").write_text(final_code, encoding="utf-8")

        # Save errors
        if not result["success"]:
            err_dir = RESULTS_DIR / "errors" / game_name
            err_dir.mkdir(parents=True, exist_ok=True)
            lines = []
            if result.get("error_stage"):
                lines.append(f"error_stage: {result['error_stage']}")
            if result.get("failed_checks"):
                lines.append("failed_checks:")
                lines.extend(f"  - {c}" for c in result["failed_checks"])
            (err_dir / f"{run_id}_{safe_cfg}.txt").write_text("\n".join(lines), encoding="utf-8")

        meta_dir = RESULTS_DIR / "meta" / game_name
        meta_dir.mkdir(parents=True, exist_ok=True)
        (meta_dir / f"{run_id}_{safe_cfg}.json").write_text(
            json.dumps(result, indent=2), encoding="utf-8"
        )
        
    return result

In [4]:
# Pre-cache structured JSON
JSON_CACHE_DIR = RESULTS_DIR / "structured_json"
JSON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Loading/generating structured JSON for all rulebooks...")
structured_cache = {}

for rulebook in RULEBOOK_FILES:
    cache_file = JSON_CACHE_DIR / rulebook.replace(".txt", ".json")

    if cache_file.exists():
        structured_cache[rulebook] = json.loads(cache_file.read_text(encoding="utf-8"))
        print(f"CACHED  {rulebook}")
        continue

    try:
        rulebook_text = (RULEBOOK_DIR / rulebook).read_text(encoding="utf-8")
        ok, structured = rule_gen.rulebook_to_json(rulebook_text)
        if ok:
            structured_cache[rulebook] = structured
            cache_file.write_text(json.dumps(structured, indent=2), encoding="utf-8")
            print(f"OK     {rulebook}")
        else:
            print(f"FAIL   {rulebook} - JSON structuring failed")
    except Exception as e:
        print(f"FAIL    {rulebook} - {e}")

Loading/generating structured JSON for all rulebooks...
CACHED  checkers.txt
CACHED  chess.txt
CACHED  connect_four.txt
CACHED  nim.txt
CACHED  reversi.txt
CACHED  tic_tac_toe.txt


In [5]:
def already_done(game_name: str, run_id: int, cfg_name: str) -> bool:
    safe_cfg = sanitize_filename(cfg_name)
    return (RESULTS_DIR / "meta" / game_name / f"{run_id}_{safe_cfg}.json").exists()

In [6]:
all_results = []
total = len(RULEBOOK_FILES) * len(CONFIGS) * RUNS_PER_GAME
completed = 0

for cfg in CONFIGS:
    apply_config(cfg)
    cfg_name = cfg["name"]

    for rulebook in RULEBOOK_FILES:
        if rulebook not in structured_cache:
            # Skips tests if JSON wasn"t able to be generated
            for run_idx in range(RUNS_PER_GAME):
                completed += 1
                
                
                print(f"[{completed}/{total}] {cfg_name} | {Path(rulebook).stem} | run {run_idx + 1} ... SKIP (no structured JSON)")
                all_results.append({
                    "rulebook_file": rulebook,
                    "game_name":     rulebook.removesuffix(".txt"),
                    "config":        cfg_name,
                    "success":       False,
                    "retries":       0,
                    "failed_checks": [],
                    "error_stage":   "skipped: JSON structuring failed",
                    "duration":      0.0,
                    "run":           run_idx + 1
                })
            continue

        for run_idx in range(RUNS_PER_GAME):
            completed += 1
            
            if already_done(Path(rulebook).stem, run_idx + 1, cfg_name):
                    print(f"[{completed}/{total}] {cfg_name} | {Path(rulebook).stem} | run {run_idx + 1} ... SKIP (already done)")
                    continue

            print(f"[{completed}/{total}] {cfg_name} | {Path(rulebook).stem} | run {run_idx + 1}", end=" ... ")

            result = run_single(rulebook, cfg_name, structured_cache[rulebook], run_idx + 1)
            result["run"] = run_idx + 1
            all_results.append(result)

            status = "OK" if result["success"] else f"FAIL ({result.get('error_stage', '')})"
            print(f"{status} | {result['duration']}s | retries: {result['retries']}")

[1/72] o4-mini + single prolog | checkers | run 1 ... SKIP (already done)
[2/72] o4-mini + single prolog | checkers | run 2 ... SKIP (already done)
[3/72] o4-mini + single prolog | checkers | run 3 ... SKIP (already done)
[4/72] o4-mini + single prolog | chess | run 1 ... SKIP (already done)
[5/72] o4-mini + single prolog | chess | run 2 ... SKIP (already done)
[6/72] o4-mini + single prolog | chess | run 3 ... SKIP (already done)
[7/72] o4-mini + single prolog | connect_four | run 1 ... SKIP (already done)
[8/72] o4-mini + single prolog | connect_four | run 2 ... SKIP (already done)
[9/72] o4-mini + single prolog | connect_four | run 3 ... SKIP (already done)
[10/72] o4-mini + single prolog | nim | run 1 ... SKIP (already done)
[11/72] o4-mini + single prolog | nim | run 2 ... SKIP (already done)
[12/72] o4-mini + single prolog | nim | run 3 ... SKIP (already done)
[13/72] o4-mini + single prolog | reversi | run 1 ... SKIP (already done)
[14/72] o4-mini + single prolog | reversi | run

---

## Analysis

In [7]:
records = []

for f in sorted(RESULTS_DIR.glob("meta/**/*.json")):
    records.append(json.loads(f.read_text(encoding="utf-8")))

df = pd.DataFrame(records)

df["model"]       = df["config"].str.extract(r"^(.+?) \+")
df["multi"]       = df["config"].str.contains("multi prolog")
df["design_plan"] = df["config"].str.contains("design plan")

df["failed_checks"] = df["failed_checks"].apply(
    lambda x: x if isinstance(x, list) else ([x] if isinstance(x, str) and x else [])
)

print(f"Runs loaded: {len(df)}")
print(f"Configs:      {df['config'].nunique()}")
print(f"Games:       {df['game_name'].nunique()}")
df.head()

Runs loaded: 144
Configs:      8
Games:       6


,rulebook_file,game_name,config,success,retries,failed_checks,duration,error_stage,model,multi,design_plan
0,checkers.txt,checkers,o4-mini + multi prolog,True,0,[],98.9,NaN,o4-mini,True,False
1,checkers.txt,checkers,o4-mini + multi prolog + design plan,True,0,[],161.0,NaN,o4-mini,True,True
2,checkers.txt,checkers,o4-mini + single prolog,True,0,[],31.4,NaN,o4-mini,False,False
3,checkers.txt,checkers,o4-mini + single prolog + design plan,True,0,[],39.6,NaN,o4-mini,False,True
4,checkers.txt,checkers,qwen3-coder:480b-cloud + multi prolog,False,0,[],36.7,max retries exceeded (no error detail),qwen3-coder:480b-cloud,True,False


In [8]:
success_by_cfg = (
    df.groupby("config")["success"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"success": "success_rate"})
    .sort_values("success_rate", ascending=False)
)

fig = px.bar(
    success_by_cfg,
    x="success_rate", y="config",
    orientation="h",
    text="success_rate",
    title="Succes rate by Config (%)",
    labels={"success_rate": "Success rate (%)", "config": "Config"},
    range_x=[0, 100]
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()

In [9]:
heatmap_data = (
    df.groupby(["game_name", "config"])["success"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .pivot(index="config", columns="game_name", values="success")
)

fig = px.imshow(
    heatmap_data,
    text_auto=True,
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=100,
    title="Surccess rate (%) - Game x Config",
    labels={"color": "Success rate (%)"},
    aspect="auto"
)
fig.update_layout(height=550)
fig.show()

In [10]:
sv_data = (
    df.groupby(["game_name", "multi"])["success"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)
sv_data["prolog_mode"] = sv_data["multi"].map({True: "Multi", False: "Single"})

fig = px.bar(
    sv_data,
    x="game_name", y="success", color="prolog_mode",
    barmode="group",
    text="success",
    title="Single vs. Multi Prolog - Success rate per game (%)",
    labels={"success": "Success rate (%)", "game_name": "Game", "prolog_mode": "Mode"},
    range_y=[0, 110]
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

In [11]:
dp_data = (
    df.groupby(["game_name", "design_plan"])["success"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)
dp_data["plan_label"] = dp_data["design_plan"].map({True: "Design Plan", False: "No Design Plan"})

fig = px.bar(
    dp_data,
    x="game_name", y="success", color="plan_label",
    barmode="group",
    text="success",
    title="Influence of Design Plans - Success rate per game (%)",
    labels={"success": "Success rate (%)", "game_name": "Game", "plan_label": "Design Plan"},
    range_y=[0, 110]
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

In [12]:
retry_df = df[df["retries"] > 0].copy()
retry_success = df[df["retries"] > 0]["success"].mean() * 100

print(f"Runs with at least 1 Retry: {len(retry_df)} ({len(retry_df)/len(df)*100:.1f}%)")
print(f"of which were successful:   {retry_success:.1f}%")
print()

retry_dist = (
    df.groupby(["config", "retries"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    retry_dist,
    x="retries", y="count", color="config",
    barmode="stack",
    title="Distribution of Retry-count per Config",
    labels={"retries": "Retry-count", "count": "Number of runs", "config": "Config"}
)
fig.show()

Runs with at least 1 Retry: 41 (28.5%)
of which were successful:   58.5%



In [13]:
from collections import Counter

failed_rows = df[df["failed_checks"].apply(len) > 0]
all_checks = [check for checks in failed_rows["failed_checks"] for check in checks]

def extract_check_label(s):
    if s.startswith("["):
        return s.split("]")[0] + "]"
    return s[:60]

counter = Counter(extract_check_label(c) for c in all_checks)
error_df = pd.DataFrame(counter.most_common(), columns=["check", "count"])

fig = px.bar(
    error_df,
    x="count", y="check",
    orientation="h",
    title="Most common failed checks (all configs & games)",
    labels={"count": "Fail count", "check": "Check"}
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450)
fig.show()

check_by_game = []
for _, row in failed_rows.iterrows():
    for check in row["failed_checks"]:
        check_by_game.append({"game_name": row["game_name"], "check": extract_check_label(check)})

cbg_df = (
    pd.DataFrame(check_by_game)
    .groupby(["game_name", "check"])
    .size()
    .reset_index(name="count")
)

fig2 = px.bar(
    cbg_df,
    x="count", y="check", color="game_name",
    orientation="h", barmode="stack",
    title="Failed checks sorted by game",
    labels={"count": "Fail count", "check": "Check", "game_name": "Game"}
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, height=450)
fig2.show()

In [14]:
consistency = (
    df.groupby(["config", "game_name"])["success"]
    .agg(["sum", "count"])
    .reset_index()
)
consistency["label"] = consistency.apply(
    lambda r: "All OK" if r["sum"] == r["count"]
    else ("All FAIL" if r["sum"] == 0
    else "Mixed"), axis=1
)

dist = (
    consistency.groupby(["config", "label"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    dist,
    x="config", y="count", color="label",
    barmode="stack",
    color_discrete_map={"All OK": "#2ecc71", "All FAIL": "#e74c3c", "Mixed": "#f39c12"},
    title="Consistency over 3 runs per config (Number of Games)",
    labels={"count": "Number of Games", "config": "Config", "label": "Result"}
)
fig.update_layout(xaxis_tickangle=-35, height=500)
fig.show()